# Imports

In [ ]:
import torch
import gc
import pandas as pd
import numpy as np
from google.colab import files
from sklearn.model_selection import train_test_split
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Load & Map Labels

In [ ]:
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]

df = pd.read_csv(CSV_PATH)
df = df.rename(columns={"article": "text"})
df["label"] = df["label"].astype(int)
df = df.dropna(subset=["text", "label"])
print(f"Loaded: {len(df)} rows")

# Train/Test Split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# Epoch Comparison

## mBERT with Epoch Count of 5

In [ ]:
import torch
import gc
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "bert-base-multilingual-cased"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True).remove_columns(["text"])
test_ds = Dataset.from_pandas(test_df).map(tokenize_function, batched=True).remove_columns(["text"])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=f"./results_{MODEL_NAME.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {"accuracy": accuracy_score(labels, predictions), "f1": f1, "precision": p, "recall": r}

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
print(f"Training {MODEL_NAME}...")
trainer.train()
results = trainer.evaluate()
print(f"Results: {results}")

## mBERT with Epoch Count of 10

In [ ]:
import torch
import gc
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "bert-base-multilingual-cased"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)

train_ds = Dataset.from_pandas(train_df).map(tokenize_function, batched=True).remove_columns(["text"])
test_ds = Dataset.from_pandas(test_df).map(tokenize_function, batched=True).remove_columns(["text"])

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

args = TrainingArguments(
    output_dir=f"./results_{MODEL_NAME.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {"accuracy": accuracy_score(labels, predictions), "f1": f1, "precision": p, "recall": r}

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
print(f"Training {MODEL_NAME}...")
trainer.train()
results = trainer.evaluate()
print(f"Results: {results}")

# Summary

Experiment 3 extended `mBERT` training to 5 and 10 epochs to assess convergence behavior. Results show that `mBERT` converged as early as epoch 3, with epoch 5 producing identical results (97.40% accuracy, 97.43% F1). At epoch 10, marginal improvement was observed (97.71% accuracy, 97.73% F1) with no signs of overfitting, confirming the stability of full fine-tuning for this task.